# Fractional cut cover in Qrisp
Build the one-hot FCC entry oracle and call it once inside Qrisp's custom block-encoding constructor. This notebook uses the interface in the [published Qrisp tutorial](https://qrisp.eu/general/tutorial/BE_tutorial/BE_vol1.html) and the [resource estimator](https://qrisp.eu/reference/Block%20Encodings/methods/resources.html).

For edge $e=(u,v)$ and vertex mask $x$, let $B_{e,x}=x_u\oplus x_v$. With $m$ edges and $n$ vertices the all-zero ancilla block is a square embedding of $B/\sqrt{m2^n}$. The class acts on a $d=\max(m,n)$ qubit operand. One-hot row $e$ occupies integer address $2^e$. Cut labels occupy columns $0,\ldots,2^n-1$. All other matrix entries are zero.

Run the cells in order. The Qrisp cells require a Python 3.11–3.13 environment. This notebook's saved outputs are from the independent gate simulation in the last section. Run the native cells to obtain the installed Qrisp version, compiler counts, and native statevector checks.

In [1]:
# Uncomment in a fresh Jupyter environment, then restart the kernel if needed.
# %pip install qrisp==0.9.10

In [2]:
import math
from importlib.metadata import version

def graph_edges(n, edges):
    if not isinstance(n, int) or n < 2:
        raise ValueError("n must be an integer of at least two")
    edges = tuple(tuple(e) for e in edges)
    if not edges:
        raise ValueError("the one-hot state requires at least one edge")
    if any(len(e) != 2 or any(not isinstance(v, int) or not 0 <= v < n for v in e)
           or e[0] == e[1] for e in edges):
        raise ValueError("supply pairs of distinct vertices numbered 0 through n-1")
    if len({tuple(sorted(e)) for e in edges}) != len(edges):
        raise ValueError("supply each undirected edge once")
    return edges

def cycle(n):
    if n < 3:
        raise ValueError("a simple cycle has at least three vertices")
    return tuple((j, (j + 1) % n) for j in range(n))

triangle = ((0, 1), (0, 2), (1, 2))

def expected_amplitude(row, col, n, edges, alpha):
    if col >= 1 << n or row == 0 or row & (row - 1):
        return 0.0
    e = row.bit_length() - 1
    if e >= len(edges):
        return 0.0
    u, v = edges[e]
    return (((col >> u) ^ (col >> v)) & 1) / alpha

## The oracle and the surrounding circuit
The gate functions below use the Qrisp primitive names. The argument `ops` is the Qrisp module in every native call. The independent test uses the same functions to record the gates.

1. Prepare a uniform superposition of the $m$ one-hot edge addresses on the scratch register.
2. Evaluate the entry predicate into a flag. Each edge temporarily borrows its second vertex bit to store endpoint parity, then restores it.
3. Flip the flag so that a crossing edge has success value zero. Apply Hadamards to the $n$ cut bits.
4. Exchange operand and scratch wires so that Qrisp's all-zero ancilla test selects the former cut register and leaves the one-hot edge on the operand. The swaps are charged in the counts.

In [3]:
def prepare_uniform_edge(edge, m, ops):
    ops.x(edge[0])
    for j in range(m - 1):
        # Controlled Ry(2 theta), followed by a CX to move the excitation.
        # For incoming |10>, the two amplitudes become cos(theta) and sin(theta).
        theta = math.acos(1 / math.sqrt(m - j))
        a, b = edge[j], edge[j + 1]
        ops.ry(theta, b)
        ops.cx(a, b)
        ops.ry(-theta, b)
        ops.cx(a, b)
        ops.cx(b, a)

def fcc_entry(edge, cut, flag_bit, edges, ops):
    """Toggle the flag by the queried edge's crossing bit."""
    for j, (u, v) in enumerate(edges):
        ops.cx(cut[u], cut[v])
        ops.mcx([edge[j], cut[v]], flag_bit, method="gray")
        ops.cx(cut[u], cut[v])

def fcc_unitary(scratch, flag, operand, n, edges, ops):
    d = max(n, len(edges))
    prepare_uniform_edge(scratch, len(edges), ops)
    fcc_entry(scratch, operand, flag[0], edges, ops)
    ops.x(flag[0])
    for j in range(n):
        ops.h(operand[j])
    for j in range(d):
        ops.swap(operand[j], scratch[j])

In [4]:
def make_fcc_block(n, edges):
    import qrisp as qr
    from qrisp.block_encodings import BlockEncoding

    edges = graph_edges(n, edges)
    m = len(edges)
    d = max(n, m)

    def unitary(scratch, flag, operand):
        fcc_unitary(scratch, flag, operand, n, edges, qr)

    return BlockEncoding(
        alpha=math.sqrt(m * (1 << n)),
        ancillas=[qr.QuantumFloat(d), qr.QuantumFloat(1)],
        unitary=unitary, num_ops=1, is_hermitian=False,
    )

## Qrisp resource estimation
The oracle-only object below represents the complete reversible entry unitary with three operand registers. The full FCC object exposes the matrix block on one operand. Both counts use the same Qrisp estimator. Its unit is one invocation. Gate counts from arbitrary rotations require a synthesis tolerance before they can be converted into fault-tolerant T counts.

In [5]:
def qrisp_resources(n, edges):
    import qrisp as qr
    from qrisp.block_encodings import BlockEncoding

    edges = graph_edges(n, edges)
    m = len(edges)
    d = max(n, m)
    def oracle_unitary(edge, cut, flag):
        fcc_entry(edge, cut, flag[0], edges, qr)

    oracle = BlockEncoding(1.0, [], oracle_unitary, num_ops=3, is_hermitian=True)
    block = make_fcc_block(n, edges)
    return {
        "version": version("qrisp"),
        "alpha": float(block.alpha),
        "entry oracle": oracle.resources(qr.QuantumFloat(m), qr.QuantumFloat(n), qr.QuantumFloat(1)),
        "FCC block": block.resources(qr.QuantumFloat(d)),
    }

for n, edges in [(3, triangle), (4, cycle(4)), (8, cycle(8))]:
    print(f"n={n}, m={len(edges)}", qrisp_resources(n, edges))

n=3, m=3 {'version': '0.8.2', 'alpha': 4.898979485566356, 'entry oracle': {'gate counts': {'t': 9, 't_dg': 12, 'h': 6, 'cx': 24}, 'depth': 30, 'qubits': 7}, 'FCC block': {'gate counts': {'cx': 39, 'ry': 4, 't': 9, 'x': 2, 'h': 9, 't_dg': 12}, 'depth': 37, 'qubits': 7}}
n=4, m=4 {'version': '0.8.2', 'alpha': 8.0, 'entry oracle': {'gate counts': {'t': 12, 't_dg': 16, 'h': 8, 'cx': 32}, 'depth': 40, 'qubits': 9}, 'FCC block': {'gate counts': {'cx': 53, 'ry': 6, 't': 12, 'x': 2, 'h': 12, 't_dg': 16}, 'depth': 47, 'qubits': 9}}
n=8, m=8 {'version': '0.8.2', 'alpha': 45.254833995939045, 'entry oracle': {'gate counts': {'t': 24, 't_dg': 32, 'h': 16, 'cx': 64}, 'depth': 80, 'qubits': 17}, 'FCC block': {'gate counts': {'cx': 109, 'ry': 14, 't': 24, 'x': 2, 'h': 24, 't_dg': 32}, 'depth': 87, 'qubits': 17}}


## Qrisp statevector verification
For each input basis vector, apply the actual block object and query the amplitudes with all signal ancillas set to zero. Check every padded row and column, including entries that should vanish. Register labels resolve the bit ordering explicitly. Exact statevector checks are intended for small graphs.

In [6]:
def check_qrisp_block(n, edges, tol=2e-5):
    import qrisp as qr
    edges = graph_edges(n, edges)
    d = max(n, len(edges))
    if d > 5:
        raise ValueError("exhaustive native verification is limited to d <= 5")
    block = make_fcc_block(n, edges)
    worst = 0.0
    for col in range(1 << d):
        operand = qr.QuantumFloat(d)
        if col:
            operand[:] = col
        scratch, flag = block.apply(operand)
        amplitude = operand.qs.statevector("function")
        for row in range(1 << d):
            observed = complex(amplitude({operand: row, scratch: 0, flag: 0}))
            target = expected_amplitude(row, col, n, edges, float(block.alpha))
            worst = max(worst, abs(observed - target))
    if worst > tol:
        raise AssertionError(f"maximum amplitude error {worst} exceeds {tol}")
    return worst

print("Triangle, maximum Qrisp amplitude error:", check_qrisp_block(3, triangle))

Triangle, maximum Qrisp amplitude error: 2.2373884051019033e-08                 
